In [2]:
from google.colab import files
uploaded = files.upload()

Saving data.csv to data.csv


In [3]:
import pandas as pd

# small dataset

data = {
    "label": [
        "spam",
        "ham",
        "spam",
        "ham",
        "spam",
        "ham",
        "spam",
        "ham"
    ],

    "text": [
        "Win free money now",
        "Hey how are you",
        "Claim your free prize",
        "Lets meet tomorrow",
        "Congratulations you won lottery",
        "Can you call me later",
        "Exclusive offer just for you",
        "See you at dinner"
    ]
}

# convert into dataframe

df = pd.DataFrame(data)

# save CSV file

df.to_csv("spam.csv", index=False)

print(df)

  label                             text
0  spam               Win free money now
1   ham                  Hey how are you
2  spam            Claim your free prize
3   ham               Lets meet tomorrow
4  spam  Congratulations you won lottery
5   ham            Can you call me later
6  spam     Exclusive offer just for you
7   ham                See you at dinner


In [4]:
!pip install pandas scikit-learn numpy joblib streamlit langchain langchain-community openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.

**SECTION 3: The ML Component — Training & Saving Models**

In [6]:
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# =========================================================
# CREATE MODELS FOLDER
# =========================================================

os.makedirs("models", exist_ok=True)

# =========================================================
# LOAD DATASETS
# =========================================================

spam_df = pd.read_csv("spam.csv")

main_df = pd.read_csv("data.csv")

print("\nSPAM DATASET:")
print(spam_df.head())

print("\nMAIN DATASET:")
print(main_df.head())

# =========================================================
# TABULAR DATASET
# TARGET COLUMN = default
# =========================================================

X = main_df.drop("default", axis=1)

y = main_df["default"]

# =========================================================
# REMOVE customer_id
# not useful for ML
# =========================================================

X = X.drop("customer_id", axis=1)

# =========================================================
# ENCODE CATEGORICAL COLUMNS
# =========================================================

for col in X.columns:

    if X[col].dtype == "object":

        le = LabelEncoder()

        X[col] = le.fit_transform(X[col])

print("\n✅ categorical encoding completed")

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================================================
# FEATURE SCALING
# =========================================================

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)

X_test_sc = scaler.transform(X_test)

print("\n✅ feature scaling complete")

# =========================================================
# MODEL REGISTRY
# =========================================================

models = {

    "svm": SVC(
        kernel="rbf",
        probability=True,
        C=1.0
    ),

    "lr": LogisticRegression(
        max_iter=500,
        C=1.0
    ),

    "rf": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

# =========================================================
# TRAIN + EVALUATE + SAVE TABULAR MODELS
# =========================================================

for name, clf in models.items():

    print(f"\n🚀 TRAINING {name.upper()} MODEL")

    clf.fit(X_train_sc, y_train)

    preds = clf.predict(X_test_sc)

    print(f"\n=== {name.upper()} REPORT ===\n")

    print(classification_report(
        y_test,
        preds
    ))

    with open(f"models/{name}_model.pkl", "wb") as f:

        pickle.dump(
            {
                "model": clf,
                "scaler": scaler
            },
            f
        )

    print(f"\n✅ {name.upper()} model saved")

# =========================================================
# SPAM DATASET
# =========================================================

X_text = spam_df["text"]

y_text = spam_df["label"]

# =========================================================
# TF-IDF VECTORIZATION
# =========================================================

vectorizer = TfidfVectorizer()

X_text_vectorized = vectorizer.fit_transform(X_text)

print("\n✅ TF-IDF vectorization complete")

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text_vectorized,
    y_text,
    test_size=0.2,
    random_state=42
)

# =========================================================
# TRAIN NAIVE BAYES MODEL
# =========================================================

spam_model = MultinomialNB(alpha=0.1)

spam_model.fit(
    X_train_text,
    y_train_text
)

print("\n✅ spam model trained")

# =========================================================
# EVALUATE SPAM MODEL
# =========================================================

spam_preds = spam_model.predict(X_test_text)

print("\n=== SPAM MODEL REPORT ===\n")

print(classification_report(
    y_test_text,
    spam_preds
))

# =========================================================
# SAVE SPAM MODEL
# =========================================================

with open("models/spam_model.pkl", "wb") as f:

    pickle.dump(
        {
            "model": spam_model,
            "vectorizer": vectorizer
        },
        f
    )

print("\n✅ SPAM MODEL SAVED")

# =========================================================
# FINAL OUTPUT
# =========================================================

print("\n🎉 ALL MODELS TRAINED & SAVED SUCCESSFULLY")

print("""
Saved Files:

models/
│
├── svm_model.pkl
├── lr_model.pkl
├── rf_model.pkl
└── spam_model.pkl
""")


SPAM DATASET:
  label                             text
0  spam               Win free money now
1   ham                  Hey how are you
2  spam            Claim your free prize
3   ham               Lets meet tomorrow
4  spam  Congratulations you won lottery

MAIN DATASET:
   customer_id  age  monthly_income employment_status  purchase_amount  \
0            1   56          151108          Salaried           122356   
1            2   46           39382     Self-Employed            44727   
2            3   32          109291          Salaried            80471   
3            4   25          126195          Salaried            10068   
4            5   38          159828     Self-Employed           118923   

   installments  previous_bnpl_loans  avg_overdue_days  missed_payments  \
0             9                    6                17                0   
1             9                    6                31                0   
2             3                    3                 2

**SECTION 4:
The Agentic Component — LangChain Tool Wrapping**

In [7]:
%%writefile ml_tools.py
# ── ml_tools.py — Agentic Multi-Model Categorization System ───────────

import pickle
import numpy as np

from langchain.tools import tool

# =========================================================
# LOAD SERIALIZED MODELS
# =========================================================

def load_model(path):

    with open(path, "rb") as f:

        return pickle.load(f)

# =========================================================
# LOAD TABULAR MODELS
# =========================================================

svm_bundle = load_model(
    "models/svm_model.pkl"
)

lr_bundle = load_model(
    "models/lr_model.pkl"
)

rf_bundle = load_model(
    "models/rf_model.pkl"
)

# =========================================================
# LOAD SPAM MODEL
# =========================================================

spam_bundle = load_model(
    "models/spam_model.pkl"
)

# =========================================================
# HELPER FUNCTION
# Used for numeric/tabular models
# =========================================================

def _infer(bundle, features):

    # convert list → numpy array
    X = np.array(features).reshape(1, -1)

    # scale features
    X_sc = bundle["scaler"].transform(X)

    # prediction
    pred = bundle["model"].predict(X_sc)[0]

    # confidence
    proba = bundle["model"].predict_proba(X_sc).max()

    return (
        f"Prediction: {pred} "
        f"(confidence: {proba:.2%})"
    )

# =========================================================
# SPAM HELPER FUNCTION
# Used for text classification
# =========================================================

def _infer_spam(text):

    # vectorize text
    vec = spam_bundle["vectorizer"].transform([text])

    # prediction
    pred = spam_bundle["model"].predict(vec)[0]

    # confidence
    proba = spam_bundle["model"].predict_proba(vec).max()

    return (
        f"Prediction: {pred} "
        f"(confidence: {proba:.2%})"
    )

# ═══════════════════════════════════════════════════════════
# TOOL DEFINITIONS
# IMPORTANT:
# The docstrings are read by the LLM
# ═══════════════════════════════════════════════════════════

# =========================================================
# SVM TOOL
# =========================================================

@tool
def classify_with_svm(features: list[float]) -> str:

    """
    Use this tool for structured numerical/tabular datasets
    such as customer financial records, income prediction,
    loan default prediction, and risk analysis.

    Best suited when numerical features are scaled and
    classes may be separated by margins.

    Input:
    List of float values.

    Returns:
    Predicted class label with confidence.
    """

    return _infer(
        svm_bundle,
        features
    )

# =========================================================
# LOGISTIC REGRESSION TOOL
# =========================================================

@tool
def classify_with_logistic(features: list[float]) -> str:

    """
    Use this tool when probability estimation and
    interpretability are important.

    Works well for balanced binary classification tasks
    and structured tabular datasets.

    Input:
    List of float values.

    Returns:
    Predicted class label with confidence.
    """

    return _infer(
        lr_bundle,
        features
    )

# =========================================================
# RANDOM FOREST TOOL
# =========================================================

@tool
def classify_with_random_forest(features: list[float]) -> str:

    """
    Use this tool for complex non-linear tabular datasets
    where many features may interact together.

    Random Forest is robust against noisy data and
    irrelevant features.

    Input:
    List of float values.

    Returns:
    Predicted class label with confidence.
    """

    return _infer(
        rf_bundle,
        features
    )

# =========================================================
# SPAM / NAIVE BAYES TOOL
# =========================================================

@tool
def classify_spam_text(text: str) -> str:

    """
    Use this tool for spam email classification,
    SMS spam detection, or text categorization tasks.

    Best suited for natural language processing tasks
    using TF-IDF text features.

    Input:
    Raw text string.

    Returns:
    Spam or ham prediction with confidence.
    """

    return _infer_spam(text)

# =========================================================
# EXPORT TOOL LIST
# =========================================================

ALL_TOOLS = [

    classify_with_svm,

    classify_with_logistic,

    classify_with_random_forest,

    classify_spam_text
]

Writing ml_tools.py


**SECTION 5: Agent Orchestration**

In [8]:
!pip install langchain langchain-community langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.7 MB/s eta 0:00:00


In [9]:
!pip install groq

In [10]:
import os

# Paste your Groq API key here

os.environ["GROQ_API_KEY"] = "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"

print("✅ GROQ API KEY SET")

✅ GROQ API KEY SET


In [11]:
# ── agent.py — Agentic Multi-Model System ───────────────────────────

from groq import Groq

from ml_tools import (
    classify_with_svm,
    classify_with_logistic,
    classify_with_random_forest,
    classify_spam_text
)

import os

# =========================================================
# STEP 1 — CREATE GROQ CLIENT
# =========================================================

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# =========================================================
# STEP 2 — TOOL REGISTRY
# Similar to ALL_TOOLS in LangChain
# =========================================================

ALL_TOOLS = {

    "svm": classify_with_svm,

    "logistic": classify_with_logistic,

    "random_forest": classify_with_random_forest,

    "spam": classify_spam_text
}

# =========================================================
# STEP 3 — REACT TOOL SELECTION
# This replaces create_react_agent()
# =========================================================

def choose_tool(user_input):

    # =====================================================
    # TEXT INPUT
    # =====================================================

    if isinstance(user_input, str):

        text = user_input.lower()

        spam_keywords = [

            "win",
            "free",
            "lottery",
            "prize",
            "money",
            "cash",
            "offer",
            "click",
            "claim"
        ]

        # spam text
        if any(word in text for word in spam_keywords):

            return "spam"

        # normal text
        return "spam"

    # =====================================================
    # NUMERICAL INPUT
    # =====================================================

    elif isinstance(user_input, list):

        return "svm"

    # =====================================================
    # UNKNOWN INPUT
    # =====================================================

    return None

# =========================================================
# STEP 4 — MAIN REACT AGENT LOOP
# Similar to AgentExecutor
# =========================================================

def classify_input(user_input):

    print("\n═══════════════════════════════")
    print("🤖 REACT AGENT STARTED")
    print("═══════════════════════════════")

    # =====================================================
    # THOUGHT
    # =====================================================

    print("\n🧠 Thought:")

    tool_name = choose_tool(user_input)

    if tool_name == "spam":

        print(
            "The input appears to be text/spam data."
        )

    elif tool_name == "svm":

        print(
            "The input appears to be numerical/tabular data."
        )

    else:

        print(
            "Could not determine input type."
        )

        return "Unknown input"

    # =====================================================
    # ACTION
    # =====================================================

    print("\n⚡ Action:")

    print(f"Selected Tool → {tool_name}")

    tool = ALL_TOOLS[tool_name]

    # =====================================================
    # OBSERVATION
    # =====================================================

    print("\n👀 Observation:")

    result = tool.invoke(user_input)

    print(result)

    # =====================================================
    # FINAL ANSWER
    # =====================================================

    print("\n✅ Final Answer:")

    final_response = f"""

Prediction Result:
{result}

Tool Used:
{tool_name}

Reason:
The agent selected the most appropriate
machine learning model based on the
input type.

"""

    print(final_response)

    return final_response

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [12]:
!pip install groq python-dotenv

In [13]:
import os

os.environ["GROQ_API_KEY"] = "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"

print("✅ GROQ API KEY SET")

✅ GROQ API KEY SET


In [14]:
os.getenv("GROQ_API_KEY")

'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'

In [15]:
from groq import Groq
import os

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("✅ GROQ CONNECTED SUCCESSFULLY")

✅ GROQ CONNECTED SUCCESSFULLY


In [24]:
%%writefile agent.py

from groq import Groq

from ml_tools import (
    classify_with_svm,
    classify_with_logistic,
    classify_with_random_forest,
    classify_spam_text
)

import os

# =========================================================
# CONFIGURE GROQ CLIENT
# =========================================================

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# =========================================================
# TOOL REGISTRY
# =========================================================

ALL_TOOLS = {

    "svm": classify_with_svm,

    "logistic": classify_with_logistic,

    "random_forest": classify_with_random_forest,

    "spam": classify_spam_text
}

# =========================================================
# TOOL SELECTION
# =========================================================

def choose_tool(user_input):

    # TEXT INPUT
    if isinstance(user_input, str):

        text = user_input.lower()

        spam_keywords = [

            "win",
            "free",
            "lottery",
            "prize",
            "money",
            "cash",
            "offer",
            "click",
            "claim"
        ]

        if any(word in text for word in spam_keywords):

            return "spam"

        return "spam"

    # NUMERICAL INPUT
    elif isinstance(user_input, list):

        return "svm"

    return None

# =========================================================
# MAIN AGENT FUNCTION
# =========================================================

def classify_input(user_input):

    tool_name = choose_tool(user_input)

    if tool_name is None:

        return "Unknown input type"

    tool = ALL_TOOLS[tool_name]

    result = tool.invoke(user_input)

    final_response = f"""

Tool Selected: {tool_name}

Prediction:
{result}

Reason:
The agent selected the best model
based on the input type.

"""

    return final_response

Writing agent.py


In [16]:
%%writefile ml_tools.py

import pickle
import numpy as np

from langchain.tools import tool

# =========================================================
# LOAD MODELS
# =========================================================

def load_model(path):

    with open(path, "rb") as f:

        return pickle.load(f)

svm_bundle = load_model(
    "models/svm_model.pkl"
)

lr_bundle = load_model(
    "models/lr_model.pkl"
)

rf_bundle = load_model(
    "models/rf_model.pkl"
)

spam_bundle = load_model(
    "models/spam_model.pkl"
)

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def _infer(bundle, features):

    X = np.array(features).reshape(1, -1)

    X_sc = bundle["scaler"].transform(X)

    pred = bundle["model"].predict(X_sc)[0]

    proba = bundle["model"].predict_proba(X_sc).max()

    return f"Prediction: {pred} (confidence: {proba:.2%})"

def _infer_spam(text):

    vec = spam_bundle["vectorizer"].transform([text])

    pred = spam_bundle["model"].predict(vec)[0]

    proba = spam_bundle["model"].predict_proba(vec).max()

    return f"Prediction: {pred} (confidence: {proba:.2%})"

# =========================================================
# TOOLS
# =========================================================

@tool
def classify_with_svm(features: list):

    return _infer(
        svm_bundle,
        features
    )

@tool
def classify_with_logistic(features: list):

    return _infer(
        lr_bundle,
        features
    )

@tool
def classify_with_random_forest(features: list):

    return _infer(
        rf_bundle,
        features
    )

@tool
def classify_spam_text(text: str):

    return _infer_spam(text)

Overwriting ml_tools.py


**SECTION 7: Streamlit Frontend — Chat Interface**

In [17]:
# ── app.py — Streamlit Interface ─────────────────────────────────────

import streamlit as st

from agent import classify_input

# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(

    page_title="Agentic Categorization System",

    page_icon="🤖",

    layout="wide"
)

# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    st.title("⚙️ Configuration")

    st.markdown("### Active Models")

    st.markdown("""
    ✅ SVM
    ✅ Logistic Regression
    ✅ Random Forest
    ✅ Spam Classifier
    """)

    st.markdown("---")

    show_reasoning = st.toggle(
        "Show Agent Reasoning",
        value=True
    )

# =========================================================
# HEADER
# =========================================================

st.title(
    "🤖 Agentic Multi-Model Categorization System"
)

st.caption(
    "ReAct Agent · Multi-Model ML System · Tool Orchestration"
)

# =========================================================
# CHAT HISTORY
# =========================================================

if "messages" not in st.session_state:

    st.session_state.messages = []

# display old messages

for msg in st.session_state.messages:

    with st.chat_message(msg["role"]):

        st.markdown(msg["content"])

# =========================================================
# USER INPUT
# =========================================================

prompt = st.chat_input(
    "Enter spam text or numerical feature vector..."
)

# =========================================================
# PROCESS INPUT
# =========================================================

if prompt:

    # save user message
    st.session_state.messages.append(

        {
            "role": "user",
            "content": prompt
        }
    )

    # show user message
    with st.chat_message("user"):

        st.markdown(prompt)

    # =====================================================
    # ASSISTANT RESPONSE
    # =====================================================

    with st.chat_message("assistant"):

        with st.spinner(
            "🤖 Agent is reasoning..."
        ):

            # =============================================
            # DETECT INPUT TYPE
            # =============================================

            try:

                # numerical vector
                if prompt.startswith("["):

                    user_input = eval(prompt)

                # text input
                else:

                    user_input = prompt

                # =========================================
                # RUN AGENT
                # =========================================

                result = classify_input(user_input)

            except Exception as e:

                result = f"❌ Error: {str(e)}"

        # ================================================
        # SHOW RESULT
        # ================================================

        st.markdown(result)

        # ================================================
        # SAVE ASSISTANT MESSAGE
        # ================================================

        st.session_state.messages.append(

            {
                "role": "assistant",
                "content": result
            }
        )

# =========================================================
# FOOTER
# =========================================================

st.markdown("---")

st.caption(
    "Built using Streamlit + Groq + ML Models"
)

2026-05-18 09:29:01.109 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 09:29:01.111 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 09:29:01.513 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-18 09:29:01.515 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 09:29:01.521 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 09:29:01.523 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 09:29:01.525 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator()

In [18]:
!pip install streamlit pyngrok

In [19]:
!streamlit run app.py &>/content/logs.txt &

In [20]:
!pip install pyngrok

In [21]:
from pyngrok import ngrok

In [22]:
ngrok.set_auth_token("3DOn5ya8m0i4Weff8JszqChLt2K_47T94FwPqcowievssCyj1")

In [23]:
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://spearfish-skewer-tadpole.ngrok-free.dev" -> "http://localhost:8501"


In [24]:
!streamlit run app.py &>/content/logs.txt &

In [25]:
%%writefile app.py

import streamlit as st

from agent import classify_input

# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(

    page_title="Agentic Categorization System",

    page_icon="🤖",

    layout="wide"
)

# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    st.title("⚙️ Configuration")

    st.markdown("### Active Models")

    st.markdown("""

✅ SVM

✅ Logistic Regression

✅ Random Forest

✅ Spam Classifier

""")

# =========================================================
# HEADER
# =========================================================

st.title(
    "🤖 Agentic Multi-Model Categorization System"
)

st.caption(
    "ReAct Agent + ML Tool Orchestration"
)

# =========================================================
# SESSION STATE
# =========================================================

if "messages" not in st.session_state:

    st.session_state.messages = []

# =========================================================
# SHOW OLD MESSAGES
# =========================================================

for msg in st.session_state.messages:

    with st.chat_message(msg["role"]):

        st.markdown(msg["content"])

# =========================================================
# USER INPUT
# =========================================================

prompt = st.chat_input(
    "Enter spam text or numerical vector..."
)

# =========================================================
# PROCESS INPUT
# =========================================================

if prompt:

    st.session_state.messages.append(

        {
            "role": "user",
            "content": prompt
        }
    )

    with st.chat_message("user"):

        st.markdown(prompt)

    with st.chat_message("assistant"):

        with st.spinner("🤖 Agent Thinking..."):

            try:

                # numerical vector
                if prompt.startswith("["):

                    user_input = eval(prompt)

                # text input
                else:

                    user_input = prompt

                result = classify_input(
                    user_input
                )

            except Exception as e:

                result = f"❌ Error: {str(e)}"

        st.markdown(result)

        st.session_state.messages.append(

            {
                "role": "assistant",
                "content": result
            }
        )

Writing app.py


In [26]:
!ls

agent.py  data.csv  ml_tools.py  __pycache__  spam.csv
app.py	  logs.txt  models	 sample_data


In [27]:
!streamlit run app.py &>/content/logs.txt &

In [28]:
!cat /content/logs.txt

In [29]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://spearfish-skewer-tadpole.ngrok-free.dev" -> "http://localhost:8501"


In [30]:
%%writefile ml_tools.py

import pickle
import numpy as np

from langchain.tools import tool

# =========================================================
# LOAD MODELS
# =========================================================

def load_model(path):

    with open(path, "rb") as f:

        return pickle.load(f)

# =========================================================
# LOAD MODEL FILES
# =========================================================

svm_bundle = load_model(
    "models/svm_model.pkl"
)

lr_bundle = load_model(
    "models/lr_model.pkl"
)

rf_bundle = load_model(
    "models/rf_model.pkl"
)

spam_bundle = load_model(
    "models/spam_model.pkl"
)

# =========================================================
# HELPER FUNCTION
# =========================================================

def _infer(bundle, features):

    X = np.array(features).reshape(1, -1)

    X_sc = bundle["scaler"].transform(X)

    pred = bundle["model"].predict(X_sc)[0]

    proba = bundle["model"].predict_proba(X_sc).max()

    return (
        f"Prediction: {pred} "
        f"(confidence: {proba:.2%})"
    )

# =========================================================
# SPAM HELPER
# =========================================================

def _infer_spam(text):

    vec = spam_bundle["vectorizer"].transform([text])

    pred = spam_bundle["model"].predict(vec)[0]

    proba = spam_bundle["model"].predict_proba(vec).max()

    return (
        f"Prediction: {pred} "
        f"(confidence: {proba:.2%})"
    )

# =========================================================
# TOOL FUNCTIONS
# =========================================================

@tool
def classify_with_svm(features: list):

    """
    Use SVM model for numerical
    and tabular classification data.
    """

    return _infer(
        svm_bundle,
        features
    )

# =========================================================

@tool
def classify_with_logistic(features: list):

    """
    Use Logistic Regression model
    for structured numerical data.
    """

    return _infer(
        lr_bundle,
        features
    )

# =========================================================

@tool
def classify_with_random_forest(features: list):

    """
    Use Random Forest model for
    complex tabular classification.
    """

    return _infer(
        rf_bundle,
        features
    )

# =========================================================

@tool
def classify_spam_text(text: str):

    """
    Use spam classifier for
    email and text spam detection.
    """

    return _infer_spam(text)

Overwriting ml_tools.py


In [31]:
!streamlit run app.py &>/content/logs.txt &

In [32]:
!cat /content/logs.txt

In [33]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://spearfish-skewer-tadpole.ngrok-free.dev" -> "http://localhost:8501"


In [34]:
%%writefile agent.py

from groq import Groq

from ml_tools import (
    classify_with_svm,
    classify_with_logistic,
    classify_with_random_forest,
    classify_spam_text
)

import os

# =========================================================
# CONFIGURE GROQ CLIENT
# =========================================================

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# =========================================================
# TOOL REGISTRY
# =========================================================

ALL_TOOLS = {

    "svm": classify_with_svm,

    "logistic": classify_with_logistic,

    "random_forest": classify_with_random_forest,

    "spam": classify_spam_text
}

# =========================================================
# TOOL SELECTION
# =========================================================

def choose_tool(user_input):

    # TEXT INPUT
    if isinstance(user_input, str):

        text = user_input.lower()

        spam_keywords = [

            "win",
            "free",
            "lottery",
            "prize",
            "money",
            "cash",
            "offer",
            "click",
            "claim"
        ]

        if any(word in text for word in spam_keywords):

            return "spam"

        return "spam"

    # NUMERICAL INPUT
    elif isinstance(user_input, list):

        return "svm"

    return None

# =========================================================
# MAIN AGENT FUNCTION
# =========================================================

def classify_input(user_input):

    tool_name = choose_tool(user_input)

    if tool_name is None:

        return "Unknown input type"

    tool = ALL_TOOLS[tool_name]

    # =====================================================
    # FIXED OBSERVATION SECTION
    # =====================================================

    # spam text
    if tool_name == "spam":

        result = tool.invoke(
            {"text": user_input}
        )

    # numerical input
    else:

        result = tool.invoke(
            {"features": user_input}
        )

    final_response = f"""

Tool Selected: {tool_name}

Prediction:
{result}

Reason:
The agent selected the best model
based on the input type.

"""

    return final_response

Overwriting agent.py


In [35]:
!streamlit run app.py &>/content/logs.txt &

In [36]:
%%writefile agent.py

from groq import Groq

from ml_tools import (
    classify_with_svm,
    classify_with_logistic,
    classify_with_random_forest,
    classify_spam_text
)

import os

# =========================================================
# CONFIGURE GROQ CLIENT
# =========================================================

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# =========================================================
# TOOL REGISTRY
# =========================================================

ALL_TOOLS = {

    "svm": classify_with_svm,

    "logistic": classify_with_logistic,

    "random_forest": classify_with_random_forest,

    "spam": classify_spam_text
}

# =========================================================
# TOOL SELECTION
# =========================================================

def choose_tool(user_input):

    # TEXT INPUT
    if isinstance(user_input, str):

        return "spam"

    # NUMERICAL INPUT
    elif isinstance(user_input, list):

        return "svm"

    return None

# =========================================================
# EXPLAIN NUMERICAL PREDICTION
# =========================================================

def explain_financial_risk(features):

    age                    = features[0]
    income                 = features[1]
    employment             = features[2]
    purchase_amount        = features[3]
    installments           = features[4]
    previous_loans         = features[5]
    overdue_days           = features[6]
    missed_payments        = features[7]

    reasons = []

    # =====================================================
    # RULE-BASED EXPLANATIONS
    # =====================================================

    if income < 60000:

        reasons.append(
            "monthly income is relatively low"
        )

    if overdue_days > 30:

        reasons.append(
            "customer has high overdue days"
        )

    if missed_payments > 0:

        reasons.append(
            "customer has missed payments history"
        )

    if previous_loans > 1:

        reasons.append(
            "customer already has multiple previous BNPL loans"
        )

    if installments > 2:

        reasons.append(
            "high number of installments increases financial burden"
        )

    if len(reasons) == 0:

        reasons.append(
            "customer profile appears financially stable"
        )

    return reasons

# =========================================================
# MAIN AGENT FUNCTION
# =========================================================

def classify_input(user_input):

    tool_name = choose_tool(user_input)

    if tool_name is None:

        return "Unknown input type"

    tool = ALL_TOOLS[tool_name]

    # =====================================================
    # SPAM TOOL
    # =====================================================

    if tool_name == "spam":

        result = tool.invoke(
            {"text": user_input}
        )

        final_response = f"""

🤖 Tool Selected: Spam Classifier

📩 Prediction:
{result}

🧠 Reasoning:
The agent detected that the input is textual data
containing spam-like language patterns such as
offers, rewards, prizes, or promotional wording.

The Spam Classifier model was selected because
it performs well for text classification tasks.

"""

    # =====================================================
    # NUMERICAL TOOL
    # =====================================================

    else:

        result = tool.invoke(
            {"features": user_input}
        )

        reasons = explain_financial_risk(
            user_input
        )

        reason_text = ""

        for r in reasons:

            reason_text += f"\n• {r}"

        final_response = f"""

🤖 Tool Selected: SVM Classifier

📊 Prediction:
{result}

🧠 Reasoning:
The agent detected structured numerical financial data,
therefore the SVM classifier was selected.

SVM works effectively for structured tabular datasets
and provides strong classification performance.

Important financial observations:
{reason_text}

📌 Features Used:
• Age
• Monthly Income
• Employment Status
• Purchase Amount
• Installments
• Previous BNPL Loans
• Overdue Days
• Missed Payments

"""

    return final_response

Overwriting agent.py


In [37]:
!streamlit run app.py &>/content/logs.txt &